In [1]:
## packages for cloud data intake: 
import s3fs
import fsspec

## packages for analysis
import pandas as pd
import xarray as xr
%config InlineBackend.figure_format = 'retina'
import seaborn as sb
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy import feature as cf
import cmocean
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import os
import colormaps as cmaps
from dask import delayed
import rioxarray as rxr
import glob
from scipy.stats import pearsonr
from tqdm import tqdm

### USEFUL FUNCTIONS

In [2]:
def cart_plot(axes,ix, ncols=3,nrows=3, rlon=-150,llon=-60,tlat=42,blat=15):
    axes.add_feature(cf.BORDERS,linewidth=0.2) #add features (boarders )
    axes.add_feature(cf.COASTLINE,linewidth=1,color='gray') #coastlines 
    gl=axes.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.01, linestyle='--') #create the map type with grids 
    axes.set_extent([llon,rlon,blat,tlat]) #set the extent of the map
    if ix%ncols ==0:
        
        gl.top_labels = False; gl.left_labels = True #set which axis labels should show 
        gl.right_labels=False; gl.xlines = False; gl.bottom_labels=False

    elif ix>=(ncols*nrows)-ncols:
        gl.top_labels = False; gl.left_labels = False #set which axis labels should show 
        gl.right_labels=False; gl.xlines = False; gl.bottom_labels=True

    else:
        gl.top_labels = False; gl.left_labels = False #set which axis labels should show 
        gl.right_labels=False; gl.xlines = False; gl.bottom_labels=False
    if ix == (ncols*nrows)-ncols:
        gl.top_labels = False; gl.left_labels = True #set which axis labels should show 
        gl.right_labels=False; gl.xlines = False; gl.bottom_labels=True
    axes.xformatter = LONGITUDE_FORMATTER;  axes.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'color':'black','size':12} #change font style for x-axis 
    gl.ylabel_style = {'color':'black','size':12}#change font style for y-axis  

convert = lambda x: chr(ord('`')+(x+1)) if x<26 else chr(ord('`')+((x)//25))+chr(ord('`')+(x+1)%26)

def precip_to_mm(da, *, interval_hours=24):
    """Convert an interval-mean precipitation flux or rate to depth in mm."""
    if interval_hours <= 0:
        raise ValueError("interval_hours must be positive.")

    raw_units = da.attrs.get("units", "")
    units = (
        raw_units.lower()
        .replace(" ", "")
        .replace("**", "")
        .replace("^", "")
        .replace("−", "-")
    )

    # Convert the input to mm/s before integrating over the interval.
    to_mm_per_second = {
        "kgm-2s-1": 1.0,       # UKESM pr
        "kg/m2/s": 1.0,
        "ms-1": 1000.0,        # WACCM PRECT
        "m/s": 1000.0,
        "mms-1": 1.0,
        "mm/s": 1.0,
        "mm/day": 1 / 86400,
        "mmday-1": 1 / 86400,
        "mmd-1": 1 / 86400,
    }
    if units not in to_mm_per_second:
        raise ValueError(f"Unsupported precipitation units: {raw_units!r}")

    factor = to_mm_per_second[units] * interval_hours * 3600
    precip = (da * factor).rename("precip")
    precip.attrs = {
        "units": "mm",
        "long_name": f"Precipitation depth over {interval_hours:g} hours",
    }
    return precip

## LOAD DATASETS 

### WACCM SSP2-4.5

In [3]:
import s3fs
import xarray as xr

_lon_slice = slice(-25, 30)
_lat_slice = slice(0, 35)

base = "reflective-persistent-prod-large/CESM2-WACCM/SSP2-4.5"
members = [f"r{i}" for i in range(6, 11)]
fs = s3fs.S3FileSystem(anon=False)


def select_region(ds):
    # Use this file's longitude coordinate and convert it to [-180, 180).
    ds = ds[["PRECT"]].assign_coords(lon=lambda d: (d.lon + 180) % 360 - 180)

    # Sort before slicing so the region spans both sides of Greenwich.
    return ds.sortby("lon").sortby("lat").sel(lon=_lon_slice, lat=_lat_slice)


member_arrays = []

for member in members:
    folder = f"{base}/{member}/day"
    prect_files = sorted(
        path for path in fs.find(folder)
        if path.endswith(".nc") and "PRECT" in path.upper()
    )
    if not prect_files:
        raise FileNotFoundError(f"No PRECT NetCDF files found in s3://{folder}/")

    # Subset each file, then combine this member's files by time coordinate.
    ds = xr.open_mfdataset(
        [fs.open(path, "rb") for path in prect_files],
        engine="h5netcdf",
        combine="by_coords",
        preprocess=select_region,
        chunks="auto",
        join="exact",
    )

    print(f"{member}: {len(prect_files)} files, {dict(ds['PRECT'].sizes)}")
    member_arrays.append(ds["PRECT"].expand_dims(member=[member]))



r6: 9 files, {'time': 31026, 'lat': 37, 'lon': 45}
r7: 6 files, {'time': 20076, 'lat': 37, 'lon': 45}
r8: 6 files, {'time': 20076, 'lat': 37, 'lon': 45}
r9: 6 files, {'time': 20076, 'lat': 37, 'lon': 45}
r10: 6 files, {'time': 20076, 'lat': 37, 'lon': 45}


In [4]:
# Require all five members to share the same dates and regional grid.
cesm_prect = xr.concat(member_arrays, dim="member")
print(cesm_prect)

/tmp/ipykernel_5053/3954142537.py:2: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  cesm_prect = xr.concat(member_arrays, dim="member")


<xarray.DataArray 'PRECT' (member: 5, time: 31026, lat: 37, lon: 45)> Size: 1GB
dask.array<concatenate, shape=(5, 31026, 37, 45), dtype=float32, chunksize=(1, 514, 37, 45), chunktype=numpy.ndarray>
Coordinates:
  * member   (member) object 40B 'r6' 'r7' 'r8' 'r9' 'r10'
  * time     (time) object 248kB 2015-01-01 00:00:00 ... 2100-01-01 00:00:00
  * lat      (lat) float64 296B 0.4712 1.414 2.356 3.298 ... 32.51 33.46 34.4
  * lon      (lon) float64 360B -25.0 -23.75 -22.5 -21.25 ... 27.5 28.75 30.0
Attributes:
    units:         m/s
    long_name:     Total (convective and large-scale) precipitation rate (liq...
    cell_methods:  time: mean


### UKESM SSP2-4.5

In [17]:
import s3fs
import xarray as xr

fs = s3fs.S3FileSystem(anon=False)
base = "reflective-persistent-prod-large/UKESM1-1/SSP245"
members = ["r3i1p1f1", "r12i1p1f1"]

_lon_slice = slice(-18, 20)  # 18°W to 20°E
_lat_slice = slice(0,35)   # 3°N to 18°N


def select_region(ds):
    # Convert longitude to [-180, 180) so the selection crosses Greenwich.
    ds = ds[["pr"]].assign_coords(lon=lambda d: (d.lon + 180) % 360 - 180)

    # Ascending coordinates allow west-to-east and south-to-north slices.
    return ds.sortby("lon").sortby("lat").sel(lon=_lon_slice, lat=_lat_slice)


member_arrays = []

for member in members:
    folder = f"{base}/{member}/ap6/day/pr"
    files = sorted(fs.glob(f"{folder}/*.nc"))
    if not files:
        raise FileNotFoundError(f"No .nc files found in s3://{folder}/")

    ds = xr.open_mfdataset(
        [fs.open(path, "rb") for path in files],
        engine="h5netcdf",
        combine="by_coords",
        preprocess=select_region,
        chunks="auto",
        join="exact",
    )

    print(f"{member}: {len(files)} files, {dict(ds['pr'].sizes)}")
    member_arrays.append(ds["pr"].expand_dims(member=[member]))

ukesm_pr = xr.concat(member_arrays, dim="member", join="exact")
print(ukesm_pr)

r3i1p1f1: 2 files, {'time': 30960, 'lat': 28, 'lon': 21}
r12i1p1f1: 2 files, {'time': 30960, 'lat': 28, 'lon': 21}
<xarray.DataArray 'pr' (member: 2, time: 30960, lat: 28, lon: 21)> Size: 146MB
dask.array<concatenate, shape=(2, 30960, 28, 21), dtype=float32, chunksize=(1, 1213, 28, 21), chunktype=numpy.ndarray>
Coordinates:
  * member   (member) object 16B 'r3i1p1f1' 'r12i1p1f1'
  * time     (time) object 248kB 2015-01-01 12:00:00 ... 2100-12-30 12:00:00
  * lat      (lat) float64 224B 0.625 1.875 3.125 4.375 ... 31.88 33.12 34.38
  * lon      (lon) float64 168B -17.81 -15.94 -14.06 ... 15.94 17.81 19.69
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    comment:        includes both liquid and solid phases
    units:          kg m-2 s-1
    original_name:  mo: (stash: m01s05i216, lbproc: 128)
    cell_methods:   area: time: mean
    cell_measures:  area: areacella


## HILLA DATASETS 

In [6]:
## Load this helper function

def open_s3_xarray_ukesm(bucket, group, scenario, variable_name, partial_filename, anon=False):
    fs = s3fs.S3FileSystem(anon=anon)
    keys = fs.glob(f"{bucket}/{group}/{scenario}/{variable_name}_{partial_filename}*.nc")
    urls = [f"simplecache::s3://{k}" for k in keys][:2]
    print(f"{bucket}/{group}/{scenario}/{variable_name}_{partial_filename}*.nc")
    ds = xr.open_mfdataset(
        urls,
        engine="h5netcdf",
        combine="by_coords",
        parallel=True,
        backend_kwargs={
            "storage_options": {
                "s3": {"anon": anon},
                "simplecache": {"cache_storage": "/tmp/s3cache"},
            }
        },
    )
    lon_names = ['longitude', 'lon']
    lon_name = [ln for ln in lon_names if ln in ds][0]
    
    lat_names = ['latitude', 'lat']
    lat_name = [lt for lt in lat_names if lt in ds][0]
    print(lat_name)
    
    # ds = ds.assign_coords({lon_name:((ds[lon_name] + 180) % 360 - 180)}).sortby(lon_name)
    ds = ds.assign_coords({lon_name: ((ds[lon_name] + 180) % 360 - 180)}).sortby(lon_name)
    if lon_name != "lon":
        ds = ds.rename({lon_name: "lon", lat_name:"lat"})
    
    return ds#.load()


## Load this helper function

def open_s3_xarray_waccm(bucket, group, scenario, variable_name, partial_filename, anon=False):
    fs = s3fs.S3FileSystem(anon=anon)
    keys = fs.glob(f"{bucket}/{group}/{scenario}{variable_name}{partial_filename}.*.nc")
    urls = [f"simplecache::s3://{k}" for k in keys]
    urls.sort()
    print(urls)
    print(f"{bucket}/{group}/{scenario}{variable_name}{partial_filename}*.nc")
    ds = [xr.open_mfdataset(
        [url],
        engine="h5netcdf",
        combine="by_coords",
        parallel=True,
        backend_kwargs={
            "storage_options": {
                "s3": {"anon": anon},
                "simplecache": {"cache_storage": "/tmp/s3cache"},
            }
        },
    )[['PRECT']] for url in urls]

    print(ds[-1])
    ds[-1] = ds[-1].sel(time=slice("2080","2084"))
    
    ds = xr.concat(ds,dim='time')
    lon_names = ['longitude', 'lon']
    lon_name = [ln for ln in lon_names if ln in ds][0]
    
    lat_names = ['latitude', 'lat']
    lat_name = [lt for lt in lat_names if lt in ds][0]
    print(lat_name)
    
    # ds = ds.assign_coords({lon_name:((ds[lon_name] + 180) % 360 - 180)}).sortby(lon_name)
    ds = ds.assign_coords({lon_name: ((ds[lon_name] + 180) % 360 - 180)}).sortby(lon_name)
    if lon_name != "lon":
        ds = ds.rename({lon_name: "lon", lat_name:"lat"})
    
    return ds

### HILLA WACCM6

In [7]:
waccm_hilla = {}
sim_time = slice("2035","2069")
_lon_slice, _lat_slice = slice(-25,30), slice(0,35)

for i in range(1, 4):
    member = f"r{i}"
    scenario = (
        f"G6-1.5k-HiLLA/{member}/ADAY/"
        f"b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.{i:03d}.cam.h1."
    )

    ds = open_s3_xarray_waccm(
        bucket="reflective-persistent-prod-large",
        group="CESM2-WACCM",
        scenario=scenario,
        variable_name="PRECT",
        partial_filename="",
    )
    waccm_hilla[member] = ds[["PRECT"]].sel(
         lon=_lon_slice, lat=_lat_slice
    )

['simplecache::s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/ADAY/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h1.PRECT.20350101-20391231.nc', 'simplecache::s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/ADAY/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h1.PRECT.20391231-20500105.nc', 'simplecache::s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/ADAY/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h1.PRECT.20500101-20600105.nc', 'simplecache::s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/ADAY/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h1.PRECT.20600101-20700105.nc', 'simplecache::s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/ADAY/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h1.PRECT.20700101-20800105.nc', 'simplecache::s3://reflective-persistent-prod-large/CESM2-WACCM/G6-1.5k-HiLLA/r1/ADAY/b.e21.BW.f09_g17.SSP245-G6-1p5K-HiLLA.001.cam.h1.PRECT.20800101-20841231.nc']
reflective-pers

In [42]:
drop_duplicate_times(list(waccm_hilla.values())[2])['PRECT']

<xarray.DataArray 'PRECT' (time: 16425, lat: 37, lon: 45)> Size: 109MB
dask.array<getitem, shape=(16425, 37, 45), dtype=float32, chunksize=(1, 37, 45), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 131kB 2035-01-01 00:00:00 ... 2084-12-31 00:00:00
  * lat      (lat) float64 296B 0.4712 1.414 2.356 3.298 ... 32.51 33.46 34.4
  * lon      (lon) float64 360B -25.0 -23.75 -22.5 -21.25 ... 27.5 28.75 30.0
Attributes:
    units:         m/s
    long_name:     Total (convective and large-scale) precipitation rate (liq...
    cell_methods:  time: mean

In [8]:
def drop_duplicate_times(ds):
    _, unique_idx = np.unique(ds.time.values, return_index=True)
    return ds.isel(time=np.sort(unique_idx))

# Stack r1, r2, and r3 along a labeled member dimension.
waccm_hilla_ensemble = xr.concat(
    [drop_duplicate_times(list(waccm_hilla.values())[0])['PRECT'],list(waccm_hilla.values())[1]['PRECT'],
    list(waccm_hilla.values())[2]['PRECT']],
    dim=xr.IndexVariable("member", list(waccm_hilla)),
).sel(time=sim_time)

# Mean PRECT at each time and grid point, requiring all three members.
waccm_hilla_mean = waccm_hilla_ensemble.mean(
    dim="member", skipna=False, keep_attrs=True
)

/tmp/ipykernel_5053/106469766.py:6: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  waccm_hilla_ensemble = xr.concat(


### HILLA UKESM1-1

In [9]:
members = ["r12i1p1f2", "r2i1p1f2", "r3i1p1f2"]
ukesm_hilla = {}
sim_time = slice("2035","2069")
_lon_slice, _lat_slice = slice(-25,30), slice(0,35)
for member in members:
    ukesm_hilla[member] = open_s3_xarray_ukesm(
        bucket="reflective-persistent-prod-large",
        group="UKESM1-1",
        scenario=f"G6-1p5K-HiLLA/{member}/apj/day/pr",
        variable_name="pr",
        partial_filename=f"day_UKESM1-1-LL_g6-1p5-hilla_{member}_gn",
    ).sel(time=sim_time, lon=_lon_slice, lat=_lat_slice)

reflective-persistent-prod-large/UKESM1-1/G6-1p5K-HiLLA/r12i1p1f2/apj/day/pr/pr_day_UKESM1-1-LL_g6-1p5-hilla_r12i1p1f2_gn*.nc


/tmp/ipykernel_5053/1956999011.py:8: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


lat
reflective-persistent-prod-large/UKESM1-1/G6-1p5K-HiLLA/r2i1p1f2/apj/day/pr/pr_day_UKESM1-1-LL_g6-1p5-hilla_r2i1p1f2_gn*.nc


/tmp/ipykernel_5053/1956999011.py:8: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


lat
reflective-persistent-prod-large/UKESM1-1/G6-1p5K-HiLLA/r3i1p1f2/apj/day/pr/pr_day_UKESM1-1-LL_g6-1p5-hilla_r3i1p1f2_gn*.nc


/tmp/ipykernel_5053/1956999011.py:8: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(


lat


In [15]:

ukesm_hilla_ensemble = xr.concat(
    [ukesm_hilla[member] for member in members],
    dim=xr.IndexVariable("member", members),
    join="exact",).sel(time=sim_time)

# Average across the three members at each time and grid point.
ukesm_hilla_mean = ukesm_hilla_ensemble.mean(
    dim="member", skipna=False, keep_attrs=True
)

In [21]:
baseline_time = slice("2020","2039")
ukesm_pr_baseline = ukesm_pr.sel(time=baseline_time).mean('member').load()
# cesm_prect_baseline = cesm_prect.sel(time=baseline_time).mean('member').load()

print('Starting for ssps')
ssp245_time = slice("2035","2069")
ukesm_pr_ssp245 = ukesm_pr.sel(time=ssp245_time).mean('member').load()
# cesm_prect_ssp245 = cesm_prect.sel(time=ssp245_time).mean('member').load()

print('Starting for HiLLA')
ukesm_hilla_ensemble = ukesm_hilla_ensemble['pr'].load()
# waccm_hilla_ensemble = waccm_hilla_ensemble.load()

Starting for ssps
Starting for HiLLA


In [22]:
ukesm_pr_baseline.to_netcdf('datasets/ukesm_pr_baseline.nc')
ukesm_pr_ssp245.to_netcdf('datasets/ukesm_pr_ssp245.nc')
ukesm_hilla_ensemble.to_netcdf('datasets/ukesm_pr_hilla.nc')

In [23]:
baseline_time = slice("2020","2039")
# ukesm_pr_baseline = ukesm_pr.sel(time=baseline_time).mean('member').load()
cesm_prect_baseline = cesm_prect.sel(time=baseline_time).mean('member').load()

print('Starting for ssps')
ssp245_time = slice("2035","2069")
# ukesm_pr_ssp245 = ukesm_pr.sel(time=ssp245_time).mean('member').load()
cesm_prect_ssp245 = cesm_prect.sel(time=ssp245_time).mean('member').load()

print('Starting for HiLLA')
# ukesm_hilla_ensemble = ukesm_hilla_ensemble['pr'].load()
waccm_hilla_ensemble = waccm_hilla_ensemble.load()

Starting for ssps
Starting for HiLLA


KeyboardInterrupt: 

In [24]:
cesm_prect_baseline.to_netcdf('datasets/cesm_prect_baseline.nc')
cesm_prect_ssp245.to_netcdf('datasets/cesm_prect_ssp245.nc')
# waccm_hilla_ensemble.to_netcdf('datasets/waccm_hilla_ensemble.nc')

In [64]:
ukesm1_pr_base = precip_to_mm(ukesm_pr_baseline)
waccm6_prect_base = precip_to_mm(cesm_prect_baseline)

ukesm1_pr_ssp = precip_to_mm(ukesm_pr_ssp245)
waccm6_prect_ssp = precip_to_mm(cesm_prect_ssp245)

ukesm1_pr_hilla = precip_to_mm(ukesm_hilla_ensemble)
waccm6_prect_hilla= precip_to_mm(waccm_hilla_ensemble)


## COMPUTE INDICES

In [73]:
import xarray as xr
import xclim as xc

def annual_indices(precip_mm):
    # Each value is a 24-hour total in mm, numerically equal to mm/day.
    pr = precip_mm.assign_attrs(units="mm/day")
    ix = xc.indices
    return xr.Dataset({
        "CWD": ix.maximum_consecutive_wet_days(pr, thresh="1 mm/day", freq="YS"),
        "CDD": ix.maximum_consecutive_dry_days(pr, thresh="1 mm/day", freq="YS"),
        "R10mm": ix.wetdays(pr, thresh="10 mm/day", freq="YS"),
        "R20mm": ix.wetdays(pr, thresh="20 mm/day", freq="YS"),
    })

inputs = {
    "ukesm_base": ukesm1_pr_base,     "waccm_base": waccm6_prect_base,
    "ukesm_ssp": ukesm1_pr_ssp,       "waccm_ssp": waccm6_prect_ssp,
    "ukesm_hilla": ukesm1_pr_hilla,   "waccm_hilla": waccm6_prect_hilla,
}
indices = {name: annual_indices(pr) for name, pr in inputs.items()}

AttributeError: module 'xarray.core' has no attribute 'ops'

In [ ]:
def annual_indices(precip_mm):
    # A 24-hour total in mm has the same numeric value as its rate in mm/day.
    pr = precip_mm.assign_attrs(units="mm/day")

    def count_days(threshold):
        return (pr >= threshold).resample(time="YS").sum().assign_attrs(units="days")

    return xr.Dataset({
        "CWD": xc.indices.maximum_consecutive_wet_days(pr, thresh="1 mm/day", freq="YS"),
        "CDD": xc.indices.maximum_consecutive_dry_days(pr, thresh="1 mm/day", freq="YS"),
        "R10mm": count_days(10),
        "R20mm": count_days(20),
    })

indices = {name: annual_indices(pr).load() for name, pr in inputs.items()}

In [78]:
# Sort the native grids and put all longitudes on the same convention.
native = {
    key: ds.assign_coords(lon=((ds.lon + 180) % 360) - 180)
           .sortby("lon").sortby("lat")
    for key, ds in indices.items()
}

# Identify the coarsest grid by its typical lat × lon spacing over the region.
def grid_spacing(ds):
    dlat = float(np.median(np.diff(ds.lat.values)))
    dlon = float(np.median(np.diff(ds.lon.values)))
    return dlat, dlon

spacing = {key: grid_spacing(ds) for key, ds in native.items()}
target_key = max(spacing, key=lambda key: np.prod(spacing[key]))
target = native[target_key]
print(f"Target grid: {target_key}, Δlat × Δlon = {spacing[target_key]} degrees")

# Keep target cell centers inside every input domain to avoid extrapolation.
def common_coord(dim):
    lower = max(float(ds[dim].min()) for ds in native.values())
    upper = min(float(ds[dim].max()) for ds in native.values())
    coord = target[dim].sel({dim: slice(lower, upper)})
    if coord.size == 0:
        raise ValueError(f"No common {dim} coordinates. Check the regional selections.")
    return coord

grid = {dim: common_coord(dim) for dim in ("lat", "lon")}

# Bilinear interpolation acts only on lat/lon and preserves time and member.
indices_common = {}
for key, ds in native.items():
    same_grid = all(ds[dim].equals(target[dim]) for dim in grid)
    indices_common[key] = (
        ds.sel(grid) if same_grid else ds.interp(grid, method="linear")
    )

Target grid: ukesm_base, Δlat × Δlon = (1.25, 1.875) degrees


In [80]:
from matplotlib.patches import Patch
import cartopy.crs as ccrs
import cartopy.feature as cfeature

models = {"ukesm": "UKESM1-1", "waccm": "CESM2-WACCM"}
comparisons = [
    ("ssp",   "hilla", "SSP − SAI"),
    ("hilla", "base",  "SAI − baseline"),
    ("ssp",   "base",  "SSP − baseline"),
]
metrics = ["CWD", "CDD", "R10mm", "R20mm"]
hatch_agreement = True


def block_weights(n, rng, n_boot=1999, block=5):
    """Weights for bootstrap means from circular blocks of consecutive years."""
    if n < 2 * block:
        raise ValueError(f"Only {n} years available for a {block}-year block bootstrap.")

    starts = rng.integers(n, size=(n_boot, (n + block - 1) // block))
    draws = (starts[..., None] + np.arange(block)) % n
    draws = draws.reshape(n_boot, -1)[:, :n]

    weights = np.zeros((n_boot, n))
    np.add.at(weights, (np.arange(n_boot)[:, None], draws), 1 / n)
    return xr.DataArray(weights, dims=("boot", "year"))


def annual_field(key, metric):
    da = indices_common[key][metric]
    if "member" not in da.dims:
        da = da.expand_dims(member=["reference"])

    # Integer years allow matching across different model calendars.
    da = da.assign_coords(year=da.time.dt.year).swap_dims({"time": "year"})
    da = da.drop_vars("time").sortby("year")
    if not np.all(np.diff(da.year.values) == 1):
        raise ValueError(f"{key}: expected one annual index per consecutive year.")
    return da


def compare(model, first, second, metric):
    a = annual_field(f"{model}_{first}", metric)
    b = annual_field(f"{model}_{second}", metric)
    paired_years = (first, second) == ("ssp", "hilla")

    if paired_years:
        years = np.intersect1d(a.year, b.year)
        a, b = a.sel(year=years), b.sel(year=years)
    elif np.intersect1d(a.year, b.year).size:
        raise ValueError("Baseline and future years overlap. Check the selected periods.")

    # Verify the spatial grids before doing arithmetic.
    a, b = xr.align(a, b, join="exact", exclude={"member", "year"})

    # Shading and sign agreement use the same analysis years.
    ac = a.mean("year", skipna=False).rename(member="member_a")
    bc = b.mean("year", skipna=False).rename(member="member_b")
    delta = ac.mean("member_a", skipna=False) - bc.mean("member_b", skipna=False)
    pairs = ac - bc
    agreement = ((pairs * delta) > 0).mean(("member_a", "member_b"))
    agreement = agreement.where(delta.notnull() & (delta != 0))

    # Bootstrap the difference between annual ensemble means.
    ay = a.mean("member", skipna=False)
    by = b.mean("member", skipna=False)
    rng = np.random.default_rng(42)
    wa = block_weights(ay.sizes["year"], rng)
    if paired_years:
        draws = xr.dot(wa, ay - by, dim="year")
    else:
        wb = block_weights(by.sizes["year"], rng)
        draws = xr.dot(wa, ay, dim="year") - xr.dot(wb, by, dim="year")

    ci = draws.chunk({"boot": -1}).quantile([0.025, 0.975], dim="boot")
    lo, hi = ci.isel(quantile=0, drop=True), ci.isel(quantile=1, drop=True)
    nonsig = ((lo <= 0) & (hi >= 0)).where(lo.notnull() & hi.notnull())

    return xr.Dataset({
        "difference": delta,
        "agreement": agreement,
        "nonsignificant": nonsig,
    }).compute()

In [84]:
import cartopy.feature as cfeature

models = {"ukesm": "UKESM1-1", "waccm": "CESM2-WACCM"}
comparisons = [
    ("ssp",   "hilla", "SSP − HiLLA"),
    ("hilla", "base",  "HiLLA − baseline"),
    ("ssp",   "base",  "SSP − baseline"),
]


def annual_mean(key, metric):
    da = indices_common[key][metric]
    if "member" in da.dims:
        da = da.mean("member", skipna=False)
    return da.assign_coords(time=da.time.dt.year).rename(time="year").sortby("year")


def bootstrap_weights(years, rng, block=5, nboot=1999):
    """Sample consecutive blocks of years and return weights for each mean."""
    n = len(years)
    if n < 2 * block or np.any(np.diff(years) != 1):
        raise ValueError(f"Need consecutive annual data and at least {2 * block} years.")

    starts = rng.integers(n, size=(nboot, (n + block - 1) // block))
    draws = (starts[..., None] + np.arange(block)) % n
    draws = draws.reshape(nboot, -1)[:, :n]

    weights = np.zeros((nboot, n))
    np.add.at(weights, (np.arange(nboot)[:, None], draws), 1 / n)
    return xr.DataArray(weights, dims=("boot", "year"), coords={"year": years})


def model_differences(model, metric):
    data = {s: annual_mean(f"{model}_{s}", metric)
            for s in ("base", "ssp", "hilla")}

    years = np.intersect1d(data["ssp"].year, data["hilla"].year)
    for s in ("ssp", "hilla"):
        data[s] = data[s].sel(year=years)

    rng = np.random.default_rng(42)
    future_w = bootstrap_weights(years, rng)
    base_w = bootstrap_weights(data["base"].year.values, rng)

    means = {s: da.mean("year", skipna=False) for s, da in data.items()}
    boot = {
        s: xr.dot(base_w if s == "base" else future_w, da, dim="year")
        for s, da in data.items()
    }
    overlap = np.intersect1d(years, data["base"].year).size > 0

    def bounds(draws, tail):
        q = draws.chunk({"boot": -1}).quantile(
            [tail, 1 - tail], dim="boot", skipna=False
        )
        return q.isel(quantile=0, drop=True), q.isel(quantile=1, drop=True)

    output = {}
    for a, b, label in comparisons:
        diff = means[a] - means[b]

        if b == "base" and overlap:
            # Two 97.5% marginal intervals give a conservative joint 95% interval.
            a_lo, a_hi = bounds(boot[a], 0.0125)
            b_lo, b_hi = bounds(boot[b], 0.0125)
            lo, hi = a_lo - b_hi, a_hi - b_lo
        else:
            lo, hi = bounds(boot[a] - boot[b], 0.025)

        output[label] = xr.Dataset({
            "diff": diff,
            "sig": (lo > 0) | (hi < 0),
        }).compute()

    return output

def plot_metric(metric):
    results = {model: model_differences(model, metric) for model in models}
    limit = max(float(abs(ds["diff"]).max()) for r in results.values() for ds in r.values())
    limit = max(limit, 0.1)
    levels = np.linspace(-limit, limit, 17)
    proj = ccrs.PlateCarree()

    fig, axes = plt.subplots(
        3, 2, figsize=(10, 9), subplot_kw={"projection": proj},
        layout="constrained",
    )
    for row, (_, _, label) in enumerate(comparisons):
        for col, (model, title) in enumerate(models.items()):
            ax = axes[row, col]
            ds = results[model][label].transpose("lat", "lon")
            shading = ax.contourf(
                ds.lon, ds.lat, ds["diff"], levels=levels, cmap="BrBG", transform=proj
            )

            # One dot at the center of each significant common-grid cell.
            iy, ix = np.where(ds["sig"].values)
            ax.scatter(ds.lon.values[ix], ds.lat.values[iy], s=4, color="black",
                       linewidths=0, transform=proj, zorder=4)

            ax.coastlines(linewidth=0.6)
            ax.add_feature(cfeature.BORDERS, linewidth=0.4)
            ax.set_title(f"{title} | {label}")

    fig.suptitle(f"{metric} · Dots mark significant differences (pointwise 95% interval)")
    fig.colorbar(shading, ax=axes, shrink=0.8, label="Difference in mean annual index (days)")
    plt.show()
    return results


results = {metric: plot_metric(metric) for metric in ("CWD", "CDD", "R10mm", "R20mm")}

KeyboardInterrupt: 